[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/AI_Engineer_Accelerator_Deep_Learning_Regularization.ipynb)


# AI Engineer Accelerator: Regularization Deep Dive

**Preventing Overfitting — The Toolkit That Makes Deep Learning Work in Practice**

| | |
|:---|:---|
| **Program** | [Emergtech AI Engineer Accelerator](https://github.com/sunilmogadati/systems-in-production) |
| **Author** | Sunil Mogadati |
| **Week** | 3-4 — Deep Learning: Training Neural Networks Effectively |
| **Also serves as** | CSCI 5931 Deep Learning — Exam 2 Study Guide (Regularization section) |


## What You Will Learn

- Why neural networks overfit and how to diagnose it (bias-variance tradeoff)
- How L1 and L2 regularization penalize large weights — and when to use which
- How dropout randomly disables neurons to build robustness
- How batch normalization stabilizes training
- Data augmentation techniques that multiply your training data for free
- Early stopping — the simplest, most effective regularization
- How to combine all techniques into a production-ready training recipe


## Key Terms

| Term | Plain English |
|:---|:---|
| **Overfitting** | Model memorizes training data; high train accuracy, low test accuracy |
| **Underfitting** | Model too simple; low accuracy on both train and test |
| **Bias** | Error from wrong assumptions (underfitting) |
| **Variance** | Error from sensitivity to training data (overfitting) |
| **L1 Regularization** | Adds sum of absolute weights to loss; creates sparsity |
| **L2 Regularization** | Adds sum of squared weights to loss; shrinks all weights (weight decay) |
| **Dropout** | Randomly disables neurons during training |
| **Batch Normalization** | Normalizes layer inputs to stable distribution |
| **Data Augmentation** | Creates new training samples by transforming existing ones |
| **Early Stopping** | Halt training when validation loss stops improving |


---
## Part I: Concept and Mental Model


## 1. The Architect's Mental Model — Why Models Fail

### The Analogy

*Imagine training a new hire to review legal contracts for risk.*

- **Underfitting:** 1-page cheat sheet with 3 rules. They miss obvious risks. Too simple.
- **Good fit:** Comprehensive training. They learn general principles. Handle new contracts well.
- **Overfitting:** Memorize 1,000 specific contracts word by word. Perfect recall on those, but lost on any new contract.

### Diagnosis Table

| Scenario | Train Acc | Test Acc | What's Wrong | Fix |
|:---|:---|:---|:---|:---|
| **Underfitting** | 70% | 68% | Model too simple | More layers/neurons, train longer |
| **Good fit** | 95% | 93% | Nothing — ship it! | — |
| **Overfitting** | 99.5% | 72% | Memorized training data | **Regularization** |


## 2. Bias-Variance Tradeoff

```
Simple Model ←――――――――――→ Complex Model
High Bias                    High Variance
Underfitting                 Overfitting
```

**Bias** = error from wrong assumptions. Model too rigid to capture the pattern.
**Variance** = error from sensitivity to training data. Model fits noise.

### Reading Loss Curves
- Both decreasing, close together → **Good fit**
- Train decreasing, val increasing → **Overfitting** (stop here!)
- Both stuck high → **Underfitting** (need more capacity)


## 3. Data Splitting Strategy

| Scenario | Training | Validation | Test |
|:---|:---|:---|:---|
| Plenty of data | 70% | 15% | 15% |
| Large dataset (1M+) | 98% | 1% | 1% |
| Limited data | K-fold CV | — | 20% |

**Always shuffle before splitting.** Test set is the final exam — evaluate ONCE.


---
## Part II: The Regularization Toolkit


## 4. L1 and L2 Regularization

*Analogy: Teacher grading accuracy AND neatness. Even correct answers lose points for huge sloppy handwriting (large weights).*

**L2 (Weight Decay):** J = Loss + (lambda/2m) * ||w||^2
- Shrinks all weights uniformly. Dense model. Default choice.

**L1 (Lasso):** J = Loss + (lambda/m) * ||w||
- Pushes some weights to exactly zero. Sparse model. Feature selection.

| Feature | L1 | L2 |
|:---|:---|:---|
| Effect | Zeroes unimportant weights | Shrinks all weights |
| Model type | Sparse | Dense |
| PyTorch | Manual | `weight_decay` in optimizer |


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# L2 in PyTorch = weight_decay parameter
model = nn.Linear(784, 10)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
print('L2 regularization = weight_decay in optimizer')
print('Common values: 1e-5 (light), 1e-4 (moderate), 1e-3 (strong)')


## 5. Dropout — Random Sabotage That Builds Strength

*Analogy: Basketball team where the coach randomly benches different players each practice. Every player must learn to perform independently. Result: robust team.*

**How it works:**
1. During training: randomly set fraction of neurons to zero
2. During inference: use ALL neurons (model.eval() disables dropout)

**Placement:**

| Location | Rate | Why |
|:---|:---|:---|
| After FC layers | 0.5 | Most parameters → highest overfitting risk |
| After Conv layers | 0.25-0.3 | Weight sharing already reduces overfitting |
| Output layer | NEVER | Need all outputs for prediction |


In [ ]:
class NetWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout_conv = nn.Dropout(0.25)  # Lower rate for conv
        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.dropout_fc = nn.Dropout(0.5)    # Higher rate for FC
        self.fc2 = nn.Linear(128, 10)         # NO dropout on output!

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.dropout_conv(x)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x

model = NetWithDropout()
print(model)
print('\nmodel.train() -> dropout ON')
print('model.eval()  -> dropout OFF')


## 6. Batch Normalization

*Analogy: Assembly line where each station receives parts from the previous. If station 2 suddenly makes parts 2x larger, station 3 breaks down. BN ensures every station receives consistently-sized parts.*

**Steps:**
1. Normalize pre-activations to mean=0, std=1
2. Scale and shift with learnable gamma and beta

**Benefits:** Stable training, higher learning rates, faster convergence, acts as regularizer

**For CNNs:** `BatchNorm2d(channels)` — normalizes per channel
**For FC:** `BatchNorm1d(features)` — normalizes per feature


In [ ]:
class NetWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)   # Per-channel normalization
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.bn2 = nn.BatchNorm1d(128)  # Per-feature normalization
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # Conv -> BN -> ReLU -> Pool
        x = x.flatten(start_dim=1)
        x = F.relu(self.bn2(self.fc1(x)))                # FC -> BN -> ReLU
        x = self.fc2(x)
        return x

print(NetWithBN())


## 7. Data Augmentation

*Analogy: Photography student with only 100 photos. Rotate, flip, crop each → 2,000+ variations. Generalizes to any orientation.*

| Technique | What It Does | When NOT to Use |
|:---|:---|:---|
| Horizontal flip | Mirror left-right | Don't flip digits (6→9!) |
| Rotation (±15 deg) | Slight rotation | Don't over-rotate buildings |
| Random crop | Cut out portion | — |
| Color jitter | Adjust brightness/contrast | — |

**Only augment training data. Never augment test data.**


In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

print('Train transforms (with augmentation):')
for t in train_transform.transforms:
    print(f'  {t}')
print('\nTest transforms (NO augmentation):')
for t in test_transform.transforms:
    print(f'  {t}')


## 8. Early Stopping

*Analogy: Cooking pasta. Check every minute. At some point it's al dente. Keep cooking → mushy (overfitting). Stop at the right moment.*

Track validation loss. When it stops decreasing for `patience` epochs → stop. Save the BEST model, not the last one.


In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, epochs=100, patience=5):
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = sum(criterion(model(img), lab).item() for img, lab in val_loader) / len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(torch.load('best_model.pth'))
    return model

print('Early stopping: save BEST model, not LAST model')


## 12. The Complete Recipe

| Technique | When to Use | PyTorch |
|:---|:---|:---|
| **L2** | Always | `weight_decay=1e-5` |
| **Dropout** | FC-heavy, small datasets | `nn.Dropout(0.5)` |
| **Batch Norm** | Nearly always | `nn.BatchNorm2d(ch)` |
| **Data Aug** | Image tasks, limited data | `transforms.Compose([...])` |
| **Early Stopping** | Always | Manual implementation |
| **More Data** | Whenever possible | Go collect |


---

## Worked Exam Problems — Regularization

### Problem 1: "Your model overfits. Walk through what each technique does."

**Scenario:** CNN on CIFAR-10. Train accuracy = 97%, Test accuracy = 68%. Gap = 29%.

**Diagnosis:** Severe overfitting. The model memorized training images.

**Apply each technique and explain the effect:**

| Technique | What You Do | What Happens to the Gap |
|:---|:---|:---|
| **L2 (weight_decay=1e-4)** | Add `weight_decay=1e-4` to optimizer. Adds λ\|\|w\|\|² to loss. | Weights shrink → model becomes simpler → gap decreases |
| **Dropout(0.5) after FC** | Add `nn.Dropout(0.5)` after each FC layer. | 50% of neurons randomly disabled each batch → forces redundancy → gap decreases |
| **BatchNorm** | Add `nn.BatchNorm2d(ch)` after each conv layer. | Stabilizes distributions + adds noise → mild regularization → gap decreases |
| **Data augmentation** | Add `RandomHorizontalFlip()`, `RandomRotation(15)` to train transforms. | Effectively increases training data → harder to memorize → gap decreases |
| **Early stopping** | Monitor val loss, stop when it rises for 5 epochs. | Prevents training past the sweet spot → gap doesn't grow |
| **Reduce model size** | Fewer filters (32→16), fewer FC neurons (512→128). | Less capacity to memorize → gap decreases, but train acc may also drop |

**Expected result after all techniques:** Train ~88%, Test ~82%. Gap = 6% (healthy).

### Problem 2: "Your model underfits. What do you do?"

**Scenario:** Train accuracy = 62%, Test accuracy = 59%. Gap = 3%.

**Diagnosis:** Underfitting. Model too simple to learn the patterns.

**What HELPS:**
- Add more layers/neurons (increase capacity)
- Train for more epochs
- Use a lower learning rate (might be overshooting the minimum)
- Try a different architecture (maybe FC for an image task → switch to CNN)

**What HURTS (do NOT do):**
- ❌ Add more dropout (reduces capacity further!)
- ❌ Increase weight_decay (restricts weights further!)
- ❌ Simplify the model (already too simple!)

### Problem 3: "Calculate L2 regularized loss"

**Given:**
- Prediction loss (cross-entropy) = 0.45
- Weights: w = [0.5, -0.3, 0.8, -0.1, 0.6]
- λ = 0.01
- m = 100 (training samples)

**Calculate:**
```
||w||² = 0.5² + (-0.3)² + 0.8² + (-0.1)² + 0.6²
       = 0.25 + 0.09 + 0.64 + 0.01 + 0.36
       = 1.35

L2 penalty = (λ / 2m) * ||w||² = (0.01 / 200) * 1.35 = 0.0000675

Total loss = 0.45 + 0.0000675 = 0.4500675
```

**The penalty is tiny** — it nudges the optimizer toward smaller weights without dominating the prediction loss. If λ were 1.0 instead of 0.01, the penalty would be 0.00675 — still small but more influential.

### Problem 4: "What does batch normalization add to the parameter count?"

**Given:** Conv2d(32→64, 3x3) followed by BatchNorm2d(64)

**Conv2d parameters:** (3 * 3 * 32 + 1) * 64 = 289 * 64 = **18,496**

**BatchNorm2d parameters:** 2 per channel (gamma + beta) = 2 * 64 = **128**
(Plus 2 running statistics per channel — mean and variance — but these are NOT learnable parameters)

**Total:** 18,496 + 128 = **18,624**

**Key insight:** BatchNorm adds negligible parameters (128 vs 18,496) but dramatically improves training.


---
## 13. Self-Check and Exam Practice

### Conceptual Questions

1. What is the difference between overfitting and underfitting? How do you diagnose each?
2. Why does L2 regularization prevent overfitting?
3. Why does dropout work? Give two explanations.
4. What does batch normalization normalize? Why are gamma and beta needed?
5. Train acc=99%, test acc=65%. What techniques would you apply?
6. Why is data augmentation applied only to training data?
7. In early stopping, why save the BEST model, not the LAST?
8. Difference between L1 and L2? When prefer L1?
9. Model underfits (train=60%, test=58%). Would more dropout help? Why not?
10. Training with batch_size=4. Should you use BatchNorm? Why or why not?

<details>
<summary>Click to reveal answers</summary>

1. Overfitting: high train, low test (gap). Underfitting: low on both.
2. ||w||^2 penalty makes large weights expensive → optimizer chooses smaller weights → simpler model.
3. (a) Ensemble of sub-networks. (b) Forces redundant representations, no co-dependency.
4. Normalizes pre-activations to mean=0, std=1. Gamma/beta let network recover original distribution if needed.
5. Data augmentation, increase dropout, increase weight_decay, early stopping, get more data.
6. Test must represent real-world conditions. Augmented test gives unrealistic accuracy.
7. Last epoch is likely overfitting. Best epoch = lowest val loss = sweet spot.
8. L1 → zeros (sparse, feature selection). L2 → shrinks (dense, general). L1 when identifying important features.
9. No! Dropout reduces capacity. Underfitting needs MORE capacity. Remove dropout, add layers.
10. Probably not. Batch stats from 4 samples are very noisy. Use GroupNorm or LayerNorm instead.

</details>
